# 03 - TF-IDF Sentiment Classification

**Goal:** predict sentiment (positive / neutral / negative) from feedback text.

**Approach:** TF-IDF bag-of-words features + Logistic Regression, wrapped in an sklearn **Pipeline** to prevent data leakage.

## Why a Pipeline matters (data leakage)

If we fit the TF-IDF vectorizer on the *whole* dataset (train + test) and *then* split, the model already "saw" the test set's word frequencies when computing IDF. This is **data leakage** and inflates results.

A `Pipeline` ensures the vectorizer is fit **only on training data** during `fit` and `cross_val_score`.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score

from src.data_loader import load_raw_tweets
from src.sentiment import prepare_sentiment_data, train_sentiment_model
from src.evaluation import evaluate_single_label, error_analysis

sns.set_theme(style="whitegrid")

In [ ]:
df = load_raw_tweets()
df = pd.DataFrame({
    "feedback": df["text"].astype(str),
    "sentiment": df["airline_sentiment"].str.lower(),
})
print(df["sentiment"].value_counts())

In [ ]:
X_train, X_test, y_train, y_test, _ = prepare_sentiment_data(
    df, text_column="feedback", label_column="sentiment"
)
print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])
print("Train classes:", sorted(set(y_train)))

## Train the baseline

TfidfVectorizer with **unigrams + bigrams** (`ngram_range=(1,2)`), then Logistic Regression.

In [ ]:
model = train_sentiment_model(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
results = evaluate_single_label(
    y_test, y_pred,
    labels=["negative", "neutral", "positive"],
    task_name="Sentiment",
)

## Interpreting the results

- **Accuracy ~0.79** is decent, but class imbalance means a model could score ~0.63 just by predicting "negative" every time.
- The **confusion matrix** shows the real behavior: negative is caught well, but neutral and positive are often confused.
- **precision/recall/F1** per class tell the real story better than accuracy alone.

## Error analysis

Let's look at actual misclassified examples to understand *why* the model fails.

In [ ]:
error_analysis(X_test, y_test, y_pred, max_examples=8);

## What the errors tell us

Common failure modes:
1. **Sarcasm / indirect phrasing** - bag-of-words can't capture tone.
2. **Neutral vs. positive ambiguity** - "thanks" can be both.
3. **Negation complexity** - "not bad" or sarcastic "great job" confuse the classifier.

## Next

The single-label airline categories do not match our app categories. So notebook `04` demonstrates the multi-label category classifier on the manually-labeled app-feedback supplement.